<!-- codex_annotation: script_overview -->
# DAPI tile 粗匹配加精匹配坐标生成

读取一张完整参考 DAPI 图和一组 tile 图像，结合 Z 字排列位置先验、低分辨率粗匹配和局部高分辨率精匹配，输出 Fiji/ImageJ 可用的 TileConfiguration.txt。

注释说明：
- 修改 FULL_IMAGE、TILE_DIR、OUTPUT_FILE 后运行。
- UPPER_ROW_MAX_INDEX 用于限制上下行搜索范围，适合规则排列的 tile。
- MATCH_THRESHOLD 低于阈值时需要人工检查坐标。


In [ ]:
# 导入需要的库
import os
import cv2
import tifffile as tiff
import numpy as np

# 限制 OpenCV 线程数，避免在 Jupyter 中占用过多资源导致 kernel 不稳定
cv2.setNumThreads(1)

# =====================================================
# 1. 路径设置
# =====================================================
FULL_IMAGE = r"D:\01.analysis\11.test_result\P4-rep2-A-merge\MAX_P4-rep2-A-0001.tif"
TILE_DIR = r"D:\01.analysis\11.test_result\P4-rep2-H\DAPI"
OUTPUT_FILE = r"D:\01.analysis\11.test_result\P4-rep2-H\DAPI\TileConfiguration.txt"

# =====================================================
# 2. 参数设置与位置先验配置
# =====================================================
MATCH_THRESHOLD = 0.50
COARSE_SCALE = 0.20
REFINE_MARGIN = 1024
COARSE_CANDIDATES = 3
UPPER_ROW_MAX_INDEX = 4  # tile_01 到该编号视为上行，其余视为下行

def normalize_to_uint8(image):
    """将原始 tif 图像归一化到 0-255，并转换为 OpenCV 适合处理的 uint8。"""
    return cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def resize_for_matching(image, scale):
    """缩小图像用于粗匹配，降低 cv2.matchTemplate 的内存占用。"""
    height, width = image.shape[:2]
    new_width = max(1, int(round(width * scale)))
    new_height = max(1, int(round(height * scale)))
    return cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_AREA)

def top_coarse_locations(result, count, min_distance):
    """从粗匹配矩阵中取多个相互分开的高分候选点。"""
    work = result.copy()
    locations = []
    for _ in range(count):
        _, score, _, loc = cv2.minMaxLoc(work)
        if not np.isfinite(score):
            break
        locations.append((loc, score))
        x, y = loc
        x0 = max(0, x - min_distance)
        y0 = max(0, y - min_distance)
        x1 = min(work.shape[1], x + min_distance + 1)
        y1 = min(work.shape[0], y + min_distance + 1)
        work[y0:y1, x0:x1] = -np.inf
    return locations

def parse_tile_number(tile_name):
    """从 tile_01.tif 这类文件名中提取 tile 编号。"""
    try:
        return int(''.join(filter(str.isdigit, os.path.splitext(tile_name)[0])))
    except ValueError:
        print(f"Warning: Cannot parse number from {tile_name}, default to 1")
        return 1

def match_tile(full_8, full_small, tile_8, tile_name):
    """结合上下行先验进行粗匹配和局部精匹配，减少 tile 跑到错误半区的风险。"""
    full_h, full_w = full_8.shape[:2]
    tile_h, tile_w = tile_8.shape[:2]
    if tile_h > full_h or tile_w > full_w:
        raise ValueError(f"Tile is larger than full image: tile={tile_8.shape}, full={full_8.shape}")

    tile_num = parse_tile_number(tile_name)
    is_upper = tile_num <= UPPER_ROW_MAX_INDEX
    mid_y = full_h // 2
    if is_upper:
        allowed_y_min = 0
        allowed_y_max = min(full_h, mid_y + tile_h)
        position_tag = "UPPER"
    else:
        allowed_y_min = max(0, mid_y - tile_h)
        allowed_y_max = full_h
        position_tag = "LOWER"

    tile_small = resize_for_matching(tile_8, COARSE_SCALE)
    coarse_result = cv2.matchTemplate(full_small, tile_small, cv2.TM_CCOEFF_NORMED)
    min_distance = max(8, min(tile_small.shape[:2]) // 4)
    candidates = top_coarse_locations(coarse_result, COARSE_CANDIDATES, min_distance)

    best = None
    for coarse_loc, coarse_score in candidates:
        coarse_x, coarse_y = coarse_loc
        guess_x = int(round(coarse_x / COARSE_SCALE))
        guess_y = int(round(coarse_y / COARSE_SCALE))
        guess_y = max(allowed_y_min, min(allowed_y_max - tile_h, guess_y))

        x_min = max(0, guess_x - REFINE_MARGIN)
        y_min = max(allowed_y_min, guess_y - REFINE_MARGIN)
        x_max = min(full_w - tile_w, guess_x + REFINE_MARGIN)
        y_max = min(allowed_y_max - tile_h, guess_y + REFINE_MARGIN)
        if y_max + tile_h <= y_min or x_max + tile_w <= x_min:
            continue

        roi = full_8[y_min:y_max + tile_h, x_min:x_max + tile_w]
        if roi.shape[0] < tile_h or roi.shape[1] < tile_w:
            continue

        refine_result = cv2.matchTemplate(roi, tile_8, cv2.TM_CCOEFF_NORMED)
        _, refine_score, _, refine_loc = cv2.minMaxLoc(refine_result)
        x = x_min + refine_loc[0]
        y = y_min + refine_loc[1]
        if best is None or refine_score > best[2]:
            best = (x, y, refine_score, coarse_score, (guess_x, guess_y), position_tag)

    if best is None:
        raise RuntimeError(f"No match candidates were found within the {position_tag} region.")
    return best

print("=" * 60)
print("Loading full image...")
print(FULL_IMAGE)
full = tiff.imread(FULL_IMAGE)
if full.ndim > 2:
    full = full[0]
full_8 = normalize_to_uint8(full)
full_small = resize_for_matching(full_8, COARSE_SCALE)
print("Original shape:", full.shape)
print("Coarse shape:", full_small.shape)

tile_files = sorted([f for f in os.listdir(TILE_DIR) if f.lower().startswith("tile_") and f.lower().endswith(".tif")])
results = []
for tile_name in tile_files:
    print("\n" + "=" * 60)
    print("Processing:", tile_name)
    tile = tiff.imread(os.path.join(TILE_DIR, tile_name))
    if tile.ndim > 2:
        tile = tile[0]
    tile_8 = normalize_to_uint8(tile)
    x, y, max_val, coarse_score, coarse_guess, pos_tag = match_tile(full_8, full_small, tile_8, tile_name)
    print(f"Assigned Row = {pos_tag}")
    print(f"Position = ({x}, {y})")
    print(f"Coarse guess = ({coarse_guess[0]}, {coarse_guess[1]})")
    print(f"Score = {max_val:.4f} coarse={coarse_score:.4f}")
    if max_val < MATCH_THRESHOLD:
        print("WARNING: Low confidence match!")
    results.append((tile_name, x, y, max_val, pos_tag))

with open(OUTPUT_FILE, "w") as f:
    f.write("dim = 2\n\n")
    for tile_name, x, y, score, _ in results:
        f.write(f"{tile_name}; ; ({x},{y})\n")

print("\nTileConfiguration saved:")
print(OUTPUT_FILE)
print("\nSummary")
for tile_name, x, y, score, pos_tag in results:
    print(f"{tile_name:12s} [{pos_tag}] x={x:6d} y={y:6d} score={score:.4f}")
print("\nDone.")
